# Stores

store 让 agent 能够跨 thread 持久化信息, 包括用户偏好、积累的知识, 以及应当超越单次对话而留存下来的事实。与 [checkpointer](https://docs.langchain.com/oss/langgraph/checkpointers) 不同, checkpointer 保存的是作用域限定在单个 thread 的完整 graph state, 而 store 保存的是可从任意 thread 访问的任意键值数据。

![Model of shared state](https://docs.langchain.com/oss/images/shared_state.png)

> **注意**
>
> **Agent Server 会自动处理 store**
>
> 使用 [Agent Server](https://docs.langchain.com/langsmith/agent-server) 时, 你无需手动实现或配置 store。API 会在幕后为你处理所有存储基础设施。

> **注意**
>
> `InMemoryStore` 适合开发与测试。在生产环境, 请使用持久化的 store, 例如 `PostgresStore`、`MongoDBStore`、`RedisStore` 或 `UpstashStore`。所有实现都继承自 `BaseStore`, 它就是在 node 函数签名中使用的类型注解。

> **注意**
>
> 可用 provider 的完整列表见 [store 集成](https://docs.langchain.com/oss/integrations/long-term-memory/index)。

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找 src/oss/langgraph/.env

# ---- 统一导入 ----
import os
import uuid

from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

## 基本用法

下面的代码片段单独展示了 `InMemoryStore`, 没有使用 LangGraph:

In [2]:
store = InMemoryStore()

memory 按 `tuple` 划分 namespace, 在下面的示例中是 `(<user_id>, "memories")`。namespace 可以是任意长度, 表示任意内容, 不必是用户特定的。

In [3]:
user_id = "1"
namespace_for_memory = (user_id, "memories")

使用 `store.put` 方法把 memory 保存到 store 中的 namespace。指定上面定义的 namespace, 以及 memory 的键值对: key 只是 memory 的一个唯一标识符 (`memory_id`), 而 value (一个字典) 就是 memory 本身。

In [4]:
memory_id = str(uuid.uuid4())
memory = {"food_preference" : "I like pizza"}
store.put(namespace_for_memory, memory_id, memory)

使用 `store.search` 方法从你的 namespace 中读出 memory, 它把给定用户的 memory 以列表形式返回, 最多为 `limit` 参数个 (默认 `10`)。使用 `InMemoryStore` 时, 条目按插入顺序返回, 因此最近的 memory 位于列表末尾; 其他 backend 可能以不同方式对 memory 排序 (见 [列出 namespace 中的条目](#列出-namespace-中的条目))。

In [5]:
memories = store.search(namespace_for_memory)
memories[-1].dict()

{'namespace': ['1', 'memories'],
 'key': '7ea73d91-3e77-4343-9ee2-5b2e667d5d82',
 'value': {'food_preference': 'I like pizza'},
 'created_at': '2026-10-01T16:32:22.703871+00:00',
 'updated_at': '2026-10-01T16:32:22.703873+00:00',
 'score': None}

每种 memory 类型都是一个 Python class ([`Item`](https://langchain-ai.github.io/langgraph/reference/store/#langgraph.store.base.Item)), 带有一些属性。我们可以用 `.dict` 把它转换成字典来访问。

它具有的属性如下:

* `value`: 该 memory 的 value (本身就是一个字典)
* `key`: 该 memory 在此 namespace 中的唯一 key
* `namespace`: 一个字符串 tuple, 即该 memory 类型的 namespace

  > **注意**
  >
  > 虽然类型是 `tuple[str, ...]`, 但当它转换为 JSON 时可能会被序列化为列表 (例如 `['1', 'memories']`)。

* `created_at`: 该 memory 创建时的时间戳
* `updated_at`: 该 memory 更新时的时间戳

## 列出 namespace 中的条目

调用 [`store.search`](https://reference.langchain.com/python/langgraph/store/#langgraph.store.base.BaseStore.search) (或 async 的 [`store.asearch`](https://reference.langchain.com/python/langgraph/store/#langgraph.store.base.BaseStore.asearch)), 且不传 `query` 也不传 `filter`, 会返回存储在 `namespace_prefix` 下的条目, 最多为 `limit` 个。当你不需要 semantic ranking 时, 可以用它来枚举某个 namespace 中的所有内容。

先向 `("alice", "memories")` 预置两条条目, 以便演示下面的枚举与分页:

In [6]:
store.put(("alice", "memories"), "mem-1", {"text": "Likes hiking"})
store.put(("alice", "memories"), "mem-2", {"text": "Dislikes loud music"})

In [7]:
# Return up to 100 items stored under ("alice", "memories").
items = store.search(("alice", "memories"), limit=100)

有三点行为需要留意:

- **`namespace_prefix` 按前缀匹配, 而非精确匹配。** `("alice",)` 也会返回 `("alice", "memories")`、`("alice", "preferences")` 等下面的条目。要限定到单一层级, 请传入完整 namespace, 或在客户端按 `item.namespace` 过滤返回的条目。
- **超过 `limit` 的结果会被静默截断。** 不会给出溢出信号 — 请把 `limit` 设为大于你预期最大值的数, 或用 `offset` 分页。
- **默认排序取决于 store backend。** `PostgresStore` 与 `AsyncPostgresStore` 返回的结果按 `updated_at` 降序排列 (最近更新的在前)。`InMemoryStore` 按插入顺序返回结果 (最近插入的在后)。不要依赖跨实现一致的特定顺序; 如果顺序重要, 请在客户端按 `item.updated_at` 排序。

要分页遍历一个很大的 namespace:

In [8]:
page_size = 50
offset = 0
while True:
    page = store.search(("alice", "memories"), limit=page_size, offset=offset)
    if not page:
        break
    for item in page:
        pass
    offset += page_size

要发现存在哪些 namespace (例如, 在列出每个用户的 memory 之前先遍历所有用户), 请使用 [`store.list_namespaces`](https://reference.langchain.com/python/langgraph/store/#langgraph.store.base.BaseStore.list_namespaces) 或 [`store.alist_namespaces`](https://reference.langchain.com/python/langgraph/store/#langgraph.store.base.BaseStore.alist_namespaces):

In [9]:
# All namespaces that start with ("alice",), truncated to two levels deep.
namespaces = store.list_namespaces(prefix=("alice",), max_depth=2)

## 语义搜索

除了简单的检索之外, store 还支持语义搜索, 让你可以基于含义而非精确匹配来查找 memory。要启用它, 请为 store 配置一个 embedding model (本示例使用硅基流动提供的 `BAAI/bge-m3`):

In [10]:
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    api_key=os.environ["SILICONFLOW_API_KEY"],
    base_url=os.environ["SILICONFLOW_BASE_URL"],
)

store = InMemoryStore(
    index={
        "embed": embeddings,                        # Embedding provider
        "dims": 1024,                               # Embedding dimensions
        "fields": ["food_preference", "$"],          # Fields to embed
    }
)

你可以通过配置 `fields` 参数, 或在存储 memory 时指定 `index` 参数, 来控制 memory 的哪些部分会被嵌入:

In [11]:
# Store with specific fields to embed
store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {
        "food_preference": "I love Italian cuisine",
        "context": "Discussing dinner plans"
    },
    index=["food_preference"]  # Only embed "food_preferences" field
)

# Store without embedding (still retrievable, but not searchable)
store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {"system_info": "Last updated: 2024-01-01"},
    index=False
)

现在搜索时, 你可以用自然语言 query 来查找相关的 memory:

In [12]:
# Find memories about food preferences
# (This can be done after putting memories into the store)
memories = store.search(
    namespace_for_memory,
    query="What does the user like to eat?",
    limit=3  # Return top 3 matches
)

## 在 LangGraph 中使用

store 与 checkpointer 协同工作: 如前所述, checkpointer 把 state 保存到 thread, 而 store 让你存储任意信息以便 _跨_ thread 访问。请按下面的方式用 checkpointer 和 store 一起编译 graph。

```python
from dataclasses import dataclass
from langgraph.checkpoint.memory import InMemorySaver

@dataclass
class Context:
    user_id: str

# We need this because we want to enable threads (conversations)
checkpointer = InMemorySaver()

# ... Define the graph ...

# Compile the graph with the checkpointer and store
builder = StateGraph(MessagesState, context_schema=Context)
# ... add nodes and edges ...
graph = builder.compile(checkpointer=checkpointer, store=store)
```

然后, 像之前一样用 `thread_id` 调用 graph, 同时也像之前一样传入 `user_id`, 它充当该特定用户的 memory 的 namespace。

```python
# Invoke the graph
config = {"configurable": {"thread_id": "1"}}

# First let's just say hi to the AI
for update in graph.stream(
    {"messages": [{"role": "user", "content": "hi"}]},
    config,
    stream_mode="updates",
    context=Context(user_id="1"),
):
    print(update)
```

通过 `Runtime` 对象, 你可以从 _任意 node_ 访问 store 和 `user_id`。当你把 `Runtime` 作为参数添加到 node 函数时, LangGraph 会自动注入它。你可以用它来保存 memory:

```python
from langgraph.runtime import Runtime
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str

async def update_memory(state: MessagesState, runtime: Runtime[Context]):

    # Get the user id from the runtime context
    user_id = runtime.context.user_id

    # Namespace the memory
    namespace = (user_id, "memories")

    # ... Analyze conversation and create a new memory

    # Create a new memory ID
    memory_id = str(uuid.uuid4())

    # We create a new memory
    await runtime.store.aput(namespace, memory_id, {"memory": memory})

```

你也可以从任意 node 访问 store, 并用 `store.search` 方法获取 memory。memory 以对象列表的形式返回, 这些对象可以转换成字典。

```python
memories[-1].dict()
{'value': {'food_preference': 'I like pizza'},
 'key': '07e0caf4-1631-47b7-b15f-65515d4c1843',
 'namespace': ['1', 'memories'],
 'created_at': '2024-10-02T17:22:31.590602+00:00',
 'updated_at': '2024-10-02T17:22:31.590605+00:00'}
```

你访问这些 memory, 并在 model 调用中使用它们。

```python
from dataclasses import dataclass
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_id: str

async def call_model(state: MessagesState, runtime: Runtime[Context]):
    # Get the user id from the runtime context
    user_id = runtime.context.user_id

    # Namespace the memory
    namespace = (user_id, "memories")

    # Search based on the most recent message
    memories = await runtime.store.asearch(
        namespace,
        query=state["messages"][-1].content,
        limit=3
    )
    info = "\n".join([d.value["memory"] for d in memories])

    # ... Use memories in the model call
```

如果你创建一个新的 thread, 只要 `user_id` 相同, 你仍然可以访问相同的 memory。

```python
# Invoke the graph on a new thread
config = {"configurable": {"thread_id": "2"}}

# Let's say hi again
for update in graph.stream(
    {"messages": [{"role": "user", "content": "hi, tell me about my memories"}]},
    config,
    stream_mode="updates",
    context=Context(user_id="1"),
):
    print(update)
```

当你在本地 (例如在 [Studio](https://docs.langchain.com/langsmith/studio) 中) 或[托管环境](https://docs.langchain.com/langsmith/platform-setup)中使用 LangSmith 时, base store 默认就可用, 你无需在编译 graph 时指定它。不过要启用语义搜索, 你**确实**需要在 `langgraph.json` 文件中配置索引设置。例如:

```json
{
    ...
    "store": {
        "index": {
            "embed": "siliconflow:BAAI/bge-m3",
            "dims": 1024,
            "fields": ["$"]
        }
    }
}
```

更多细节与配置选项见[部署指南](https://docs.langchain.com/langsmith/semantic-search)。

## 构建自定义 store

要使用内置实现之外的存储 backend, 请继承 `BaseStore` 并实现其必需的方法。内置的 `InMemoryStore` 是最简单的参考实现。

### 基础契约

五个 async 方法都是必需的。对应的同步方法 (`put`、`get`、`delete`、`search`、`list_namespaces`) 是可选的, 但为了兼容同步的 graph 执行, 建议实现它们。

| 方法 | 说明 |
|--------|-------------|
| `aput(namespace, key, value, index=None)` | 存储或覆盖单个条目 |
| `aget(namespace, key)` | 按键检索单个条目; 若不存在则返回 `None` |
| `adelete(namespace, key)` | 删除单个条目 |
| `asearch(namespace_prefix, *, query=None, filter=None, limit=10, offset=0)` | 在某个 namespace 前缀下搜索条目; 可选地按语义 query 搜索 |
| `alist_namespaces(*, prefix=None, suffix=None, max_depth=None, limit=100, offset=0)` | 列出匹配某前缀/后缀模式的 namespace |

实现前请先查阅准确的签名:

In [13]:
import inspect
from langgraph.store.base import BaseStore
print(inspect.getsource(BaseStore))

class BaseStore(ABC):
    """Abstract base class for persistent key-value stores.

    Stores enable persistence and memory that can be shared across threads,
    scoped to user IDs, assistant IDs, or other arbitrary namespaces.
    Some implementations may support semantic search capabilities through
    an optional `index` configuration.

    Note:
        Semantic search capabilities vary by implementation and are typically
        disabled by default. Stores that support this feature can be configured
        by providing an `index` configuration at creation time. Without this
        configuration, semantic search is disabled and any `index` arguments
        to storage operations will have no effect.

        Similarly, TTL (time-to-live) support is disabled by default.
        Subclasses must explicitly set `supports_ttl = True` to enable this feature.
    """

    supports_ttl: bool = False
    ttl_config: TTLConfig | None = None

    __slots__ = ("__weakref__",)

    @abstract

### namespace 设计

namespace 是字符串 tuple, 例如 `("user_id", "memories")`。store 实现必须支持:

- **前缀匹配**: `asearch(("alice",))` 会返回 `("alice",)`、`("alice", "memories")` 以及任何其他子 namespace 下的条目。
- **精确 key 查找**: `aget(("alice", "memories"), "some-key")` 必须是 O(1) 或接近 O(1)。

对于 SQL backend, 一种常见 schema:

```sql
CREATE TABLE store_items (
    namespace   TEXT[] NOT NULL,
    key         TEXT NOT NULL,
    value       JSONB NOT NULL,
    created_at  TIMESTAMPTZ DEFAULT now(),
    updated_at  TIMESTAMPTZ DEFAULT now(),
    PRIMARY KEY (namespace, key)
);

CREATE INDEX ON store_items USING gin(namespace);
```

### 序列化

store 的 value 是普通的 Python dict — 不需要特殊的序列化器。用 `json.dumps` / `json.loads` 序列化, 或直接使用 JSONB 列。不要存储无法 JSON 序列化的原始 Python 对象。

### 语义搜索支持

如果你的 backend 支持向量搜索, 请在 `asearch` 上实现 `query` 参数:

- 接受一个 `query: str | None` 参数。
- 当 `query` 不为 `None` 时, 对它进行嵌入, 并按余弦相似度对结果排序。
- 当提供了 `query` 时, 每个 `Item` 上都应包含一个 `score` 字段。

如果你的 backend 不支持向量搜索, 请在传入 `query` 时抛出 `NotImplementedError`。

### 测试

目前还没有针对自定义 store 的一致性测试套件。请以 `InMemoryStore` 为参考进行测试:

```python
import pytest
from langgraph.store.memory import InMemoryStore
from your_module import YourStore

@pytest.fixture
async def store():
    async with YourStore.create() as s:
        yield s

@pytest.fixture
def reference():
    return InMemoryStore()

async def test_put_and_get(store, reference):
    ns = ("test", "ns")
    for s in [store, reference]:
        await s.aput(ns, "k1", {"val": 1})
        item = await s.aget(ns, "k1")
        assert item is not None
        assert item.value == {"val": 1}

async def test_delete(store, reference):
    ns = ("test", "ns")
    for s in [store, reference]:
        await s.aput(ns, "k1", {"val": 1})
        await s.adelete(ns, "k1")
        assert await s.aget(ns, "k1") is None

async def test_search_prefix(store, reference):
    for s in [store, reference]:
        await s.aput(("user", "memories"), "m1", {"text": "likes pizza"})
        results = await s.asearch(("user",))
        assert any(r.key == "m1" for r in results)
```

### 后续步骤

- [向 Agent Server 添加自定义 store](https://docs.langchain.com/langsmith/custom-store) — 部署你的实现
- [Checkpointers](https://docs.langchain.com/oss/langgraph/checkpointers) — thread 作用域的 state persistence